# Build Session-Aware Assistant

This notebook creates a simple Azure AI Foundry assistant that remembers context inside each session. The notebook keeps a local mapping from a friendly `session_id` such as `customer-a` to an Azure AI Foundry conversation ID.

Two users can talk to the same assistant, but each user gets a separate conversation history.

## 1. Install required packages

Run this once if your environment does not already have these packages.

In [ ]:
%pip install azure-ai-projects==2.0.0b2 azure-identity python-dotenv

## 2. Load Azure AI Foundry configuration

The notebook looks for `.env` in this order:

1. This notebook folder
2. `../A2A_and_MCP/.env`
3. `../Getting_Started_Foundry_Agent/.env`

Expected values:

```text
FOUNDRY_PROJECT_ENDPOINT="https://...services.ai.azure.com/api/projects/..."
MODEL_DEPLOYMENT_NAME="your-model-deployment-name"
```

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity.aio import DefaultAzureCredential
from azure.ai.projects.aio import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

cwd = Path.cwd()

candidate_env_paths = [
    cwd / ".env",
    cwd / "Memory-Build session-aware assistant" / ".env",
    cwd / "A2A_and_MCP" / ".env",
    cwd / "Getting_Started_Foundry_Agent" / ".env",
    cwd.parent / "A2A_and_MCP" / ".env",
    cwd.parent / "Getting_Started_Foundry_Agent" / ".env",
]

loaded_env_path = None
for env_path in candidate_env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=False)
        loaded_env_path = env_path
        break

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

if not foundry_project_endpoint or not model_deployment_name:
    raise ValueError(
        "Missing FOUNDRY_PROJECT_ENDPOINT or MODEL_DEPLOYMENT_NAME. "
        "Create a .env file or update one of the existing repo .env files."
    )

print(f"Loaded configuration from: {loaded_env_path}")
print(f"Foundry project endpoint: {foundry_project_endpoint}")
print(f"Model deployment name: {model_deployment_name}")

## 3. Connect to the Azure AI Foundry project

This uses your Azure sign-in through `DefaultAzureCredential`. If authentication fails, run `az login` in a terminal and rerun this cell.

In [ ]:
credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=credential,
)

print("Connected to Azure AI Foundry project.")

## 4. Create a session-aware assistant agent

The agent itself gives the assistant its behavior. Session memory comes from using the same Foundry conversation ID for repeated turns in the same session.

In [ ]:
agent_name = "session-aware-assistant"

agent_instructions = """
You are a session-aware training assistant for Azure AI Foundry learners.
Remember facts shared earlier in the same conversation.
Do not assume facts from other users or other sessions.
When useful, summarize what you remember about the current session before answering.
Keep answers practical and concise.
"""

agent = await project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions=agent_instructions,
    ),
)

print(f"Agent created: id={agent.id}, name={agent.name}, version={agent.version}")

## 5. Build the session manager

`SessionAwareAssistant` keeps a dictionary like this:

```python
{
    "customer-a": "foundry-conversation-id-1",
    "customer-b": "foundry-conversation-id-2",
}
```

That is the main wiring: same `session_id` means same Foundry conversation, so the assistant remembers earlier turns in that session.

In [ ]:
class SessionAwareAssistant:
    def __init__(self, project_client: AIProjectClient, agent_name: str):
        self.project_client = project_client
        self.agent_name = agent_name
        self.openai_client = project_client.get_openai_client()
        self.sessions: dict[str, str] = {}

    async def get_or_create_conversation_id(self, session_id: str) -> str:
        if session_id not in self.sessions:
            conversation = await self.openai_client.conversations.create()
            self.sessions[session_id] = conversation.id
            print(f"Created new Foundry conversation for session '{session_id}': {conversation.id}")
        return self.sessions[session_id]

    async def ask(self, session_id: str, user_message: str) -> str:
        conversation_id = await self.get_or_create_conversation_id(session_id)

        response = await self.openai_client.responses.create(
            conversation=conversation_id,
            extra_body={
                "agent": {
                    "name": self.agent_name,
                    "type": "agent_reference",
                }
            },
            input=user_message,
        )

        return response.output_text

    def list_sessions(self) -> dict[str, str]:
        return dict(self.sessions)

    def reset_session(self, session_id: str) -> None:
        self.sessions.pop(session_id, None)


assistant = SessionAwareAssistant(project_client, agent_name)
print("Session-aware assistant is ready.")

## 6. Test one session with memory

The assistant should remember the user's name and learning goal because both turns use `session_id="learner-1"`.

In [ ]:
reply_1 = await assistant.ask(
    session_id="learner-1",
    user_message="My name is Ajay. I am learning Azure AI Foundry agents for a Level 3 training session.",
)

print(reply_1)

In [ ]:
reply_2 = await assistant.ask(
    session_id="learner-1",
    user_message="What do you remember about me, and what should I learn next?",
)

print(reply_2)

## 7. Test a second isolated session

This uses `session_id="learner-2"`, so the assistant should not know Ajay's earlier details unless they are repeated in this session.

In [ ]:
reply_3 = await assistant.ask(
    session_id="learner-2",
    user_message="What do you know about my name and training goal?",
)

print(reply_3)

## 8. Inspect and reset sessions

This is useful in demos because you can show exactly which local session keys are mapped to Foundry conversation IDs.

In [ ]:
assistant.list_sessions()

In [ ]:
# Optional: reset one session so the next message starts a fresh Foundry conversation.
# assistant.reset_session("learner-1")
# assistant.list_sessions()

## 9. Optional cleanup

Run this when you are done with the notebook.

In [ ]:
if hasattr(project_client, "close"):
    await project_client.close()

if hasattr(credential, "close"):
    await credential.close()

print("Closed Azure clients.")

## Wiring Summary

```mermaid
flowchart LR
    User[User message] --> SessionId[session_id]
    SessionId --> Lookup{Known session?}
    Lookup -->|No| NewConversation[Create Foundry conversation]
    Lookup -->|Yes| ExistingConversation[Reuse Foundry conversation]
    NewConversation --> Responses[responses.create]
    ExistingConversation --> Responses
    Responses --> Agent[session-aware-assistant agent]
    Agent --> Answer[Assistant response]
```

The important idea is simple: the assistant becomes session-aware because each `session_id` consistently reuses the same Azure AI Foundry conversation ID.